# Site annotation
`sfsutils` can add site-level annotations such as the ancestral allele, site degeneracy, and synonymy. The {class}`~sfsutils.parser.Parser` applies them on the fly while it builds a spectrum, while the {class}`~sfsutils.annotation.Annotator` applies them and writes the annotated variants to a file. Polarising against an ancestral state is only needed for an unfolded spectrum: the {class}`~sfsutils.parser.Parser` reads it from the `AA` field by default (this can be customised), and otherwise the spectrum can be folded.

## Degeneracy Annotation
{class}`~sfsutils.annotation.DegeneracyAnnotation` annotates the SFS by the degeneracy of the site. This annotation requires information from a FASTA and GFF file and is useful for stratifying the SFS by 0-fold and 4-fold degenerate sites, a common way of contrasting putatively neutral and selected sites (see {class}`~sfsutils.parser.DegeneracyStratification`).

In [ ]:
setwd("~/PycharmProjects/SFSUtils/")

In [ ]:
library(sfsutils)

su <- load_sfsutils()

In [ ]:
# example for degeneracy annotation
ann <- su$Annotator(
  source = "resources/genome/betula/biallelic.subset.10000.vcf.gz",
  fasta = "resources/genome/betula/genome.subset.20.fasta",
  gff = "resources/genome/betula/genome.gff.gz",
  annotations = list(su$DegeneracyAnnotation()),
  output = "genome.deg.vcf.gz"
)

ann$annotate()

## Ancestral Allele Annotation
Currently, two ancestral allele annotations are available: {class}`~sfsutils.annotation.MaximumParsimonyAncestralAnnotation` and {class}`~sfsutils.annotation.MaximumLikelihoodAncestralAnnotation`. The former is error-prone and not recommended. Alternatively, if outgroups are missing, the spectra can be folded, though this discards information and yields a less informative spectrum. Ideally, we would like to use {class}`~sfsutils.annotation.MaximumLikelihoodAncestralAnnotation`, which is more sophisticated and requires one or several outgroup to be specified. Its underlying model is based on [`EST-SFS`](https://doi.org/10.1534/genetics.118.301120). The maximum-likelihood model estimates branch rates from monomorphic sites, so ideally these are present in the input. When there are few or none, supply {attr}`~sfsutils.annotation.MaximumLikelihoodAncestralAnnotation.n_target_sites`, the total number of sites (segregating and monomorphic) underlying the data, along with a FASTA reference from which the monomorphic sites are sampled.

In [ ]:
# example for ancestral allele annotation with outgroups
ann <- su$Annotator(
  source = "resources/genome/betula/all.with_outgroups.subset.10000.vcf.gz",
  fasta = "resources/genome/betula/genome.subset.20.fasta",
  annotations = list(su$MaximumLikelihoodAncestralAnnotation(
    outgroups = c("ERR2103730"),
    n_ingroups = 10,
    n_target_sites = 200000
  )),
  output = "genome.aa.vcf.gz"
)

ann$annotate()